In [1]:
import tarfile
import gzip
import io
import json

archive_path = "data/newsroom-release.tar"


def print_examples(split_name, n=3):
    with tarfile.open(archive_path, "r") as tf:
        member = None
        for m in tf.getmembers():
            if m.name.endswith(f"{split_name}.jsonl.gz"):
                member = m
                break

        if member is None:
            print(f"No {split_name} split found")
            return

        fh = tf.extractfile(member)
        if fh is None:
            print(f"Could not read {split_name} split")
            return

        with gzip.GzipFile(fileobj=fh, mode="rb") as gz:
            for i, line in enumerate(gz, start=1):
                if i > n:
                    break
                row = json.loads(line)
                print(f"\n=== {split_name.upper()} example {i} ===")
                print("title:", row.get("title", ""))
                print("date:", row.get("date", ""))
                print("summary:", row.get("summary", ""))
                text = row.get("text", "")
                print("text preview:", text[:400].replace("\n", " "))
                print("-" * 80)


for split in ["dev", "test", "train"]:
    print_examples(split)



=== DEV example 1 ===
title: Pro Sports Xchange notes
date: 19980117162148
summary: SAN DIEGO PADRES team notebook
text preview: So sayeth Padre general manager Kevin Towers.  Less than two weeks after unofficially setting his starting rotation by trading for No. 1 Kevin Brown and re-signing No. 5 Pete Smith, Towers stirred the pot Jan. 7 by signing Mark Langston to a minor league contract.  But the Padres have no intention of having Langston pitch at Triple-A Las Vegas. And neither does the 37-year-old left-hander.  "We di
--------------------------------------------------------------------------------

=== DEV example 2 ===
title: India Becoming a Crucial Cog in the Machine at I.B.M.
date: 20060620021852
summary: India provides I.B.M. with its fastest-growing market and a crucial base for delivering services to much of the world.
text preview: BANGALORE, India, June 4  The world's biggest computer services company could not have chosen a more appropriate setting to lay out its stra

In [2]:
import tarfile
import gzip
import json
import re
from collections import Counter

# Path to your dataset
TAR_PATH = "data/newsroom-release.tar"

# Files inside the tar archive
FILES = [
    "release/train.jsonl.gz",
    "release/dev.jsonl.gz",
    "release/test.jsonl.gz",
]

# Simple tokenizer
WORD_RE = re.compile(r"\b[a-zA-Z][a-zA-Z'-]*\b")


def tokenize(text):
    return WORD_RE.findall(text.lower())


def process_text(text, bigrams, trigrams):
    words = tokenize(text)

    # Bigrams
    for i in range(len(words) - 1):
        gram = words[i:i+2]
        if "solar" in gram:
            bigrams[" ".join(gram)] += 1

    # Trigrams
    for i in range(len(words) - 2):
        gram = words[i:i+3]
        if "solar" in gram:
            trigrams[" ".join(gram)] += 1


def main():
    bigrams = Counter()
    trigrams = Counter()
    print("Starting script")

    print("Opening tar...")

    with tarfile.open(TAR_PATH, "r") as tar:
        print("Tar opened successfully")

        for filename in FILES:
            print(f"Processing {filename}")

            member = tar.getmember(filename)

            with tar.extractfile(member) as raw_file:
                with gzip.open(raw_file, "rt", encoding="utf-8") as gz:

                    for line_num, line in enumerate(gz, 1):
                        try:
                            obj = json.loads(line)
                        except json.JSONDecodeError:
                            continue

                        # Process common text fields if present
                        for field in ("text", "summary", "title"):
                            if field in obj and obj[field]:
                                process_text(obj[field], bigrams, trigrams)

                        if line_num % 10000 == 0:
                            print(f"  {line_num:,} documents")

    # Save bigrams
    with open("solar_bigrams.txt", "w", encoding="utf-8") as f:
        f.write("Count\tPhrase\n")
        for phrase, count in bigrams.most_common():
            f.write(f"{count}\t{phrase}\n")

    # Save trigrams
    with open("solar_trigrams.txt", "w", encoding="utf-8") as f:
        f.write("Count\tPhrase\n")
        for phrase, count in trigrams.most_common():
            f.write(f"{count}\t{phrase}\n")

    print(f"Found {len(bigrams)} unique bigrams.")
    print(f"Found {len(trigrams)} unique trigrams.")
    print("Results saved to:")
    print("  solar_bigrams.txt")
    print("  solar_trigrams.txt")


if __name__ == "__main__":
    main()

Starting script
Opening tar...
Tar opened successfully
Processing release/train.jsonl.gz
  10,000 documents
  20,000 documents
  30,000 documents
  40,000 documents
  50,000 documents
  60,000 documents
  70,000 documents
  80,000 documents
  90,000 documents
  100,000 documents
  110,000 documents
  120,000 documents
  130,000 documents
  140,000 documents
  150,000 documents
  160,000 documents
  170,000 documents
  180,000 documents
  190,000 documents
  200,000 documents
  210,000 documents
  220,000 documents
  230,000 documents
  240,000 documents
  250,000 documents
  260,000 documents
  270,000 documents
  280,000 documents
  290,000 documents
  300,000 documents
  310,000 documents
  320,000 documents
  330,000 documents
  340,000 documents
  350,000 documents
  360,000 documents
  370,000 documents
  380,000 documents
  390,000 documents
  400,000 documents
  410,000 documents
  420,000 documents
  430,000 documents
  440,000 documents
  450,000 documents
  460,000 documents
